# 01 — Veri Hazırlama ve Etiket Üretimi

Bu notebook, çalışmanın **ham veriden eğitilebilir veri kümesine** dönüşümünü adım adım
gerçekleştirir. Tüm görüntü işleme adımları burada toplanmıştır; sonraki notebook'lar
doğrudan burada üretilen çıktıları kullanır.

## Akış

| # | Adım | Girdi | Çıktı |
|---|------|-------|-------|
| 1 | DICOM → PNG dönüşümü, `InstanceNumber` sırasına göre | DICOM arşivi | `png/` |
| 2 | `annotations.json` → orijinal görüntü + ikili maske | `annotations.json` + `png/` | `originals/`, `masks/` |
| 3 | Maskeyi kırmızıya boyama (etiket üretimi için ara adım) | `masks/` | `red_masks/` |
| 4 | Maske → YOLO poligon etiketi | `red_masks/` | `labels/` |
| 5 | Eğitim / doğrulama ayrımı | `originals/` + `labels/` | `dataset/` |

## Girdi biçimleri

* **DICOM arşivi** — Hastaların aksiyel kesitleri. Kesit sırası `InstanceNumber`
  etiketiyle belirlenir; bu etiket olmadan kesitler anatomik olarak yanlış sırada okunur.
* **`annotations.json`** — COCO benzeri yapı:

  ```json
  {
    "images":      [{"id": 1, "file_name": "hasta001.png", "width": 512, "height": 512}],
    "annotations": [{"image_id": 1, "segmentation": [[x1, y1, x2, y2, "..."]], "area": 1234, "bbox": [x, y, w, h]}]
  }
  ```

  Segmentasyon poligonu düz bir koordinat listesidir. Tüm lezyonlar **tek sınıf** altında
  toplanmıştır: `Cancer`.

> **Mahremiyet notu:** Ham DICOM verisi hasta verisi içerdiği için bu depoda yer almaz.
> Burada yalnızca ondan türeyen veri kümesinin nasıl oluşturulduğu belgelenmiştir.


## 0. Ortam ve yollar

Tüm yollar tek bir yerde tanımlıdır. Colab dışında çalışıyorsanız yalnızca bu hücreyi
düzenlemeniz yeterlidir.


In [ ]:
# Colab kullanıyorsanız Drive'ı bağlayın:
# from google.colab import drive
# drive.mount("/content/drive")

# Gerekli paketler (Colab dışında):
# %pip install -q pydicom opencv-python-headless scikit-learn

import os
import json
import shutil

import cv2
import numpy as np
import pydicom
from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------- yollar
DATA_ROOT = "/content/drive/MyDrive/lungscan"          # türetilmiş verinin kök dizini
DICOM_ROOT = "/content/drive/MyDrive/kaynak_klasor"    # ham DICOM arşivi (hasta verisi)

PNG_DIR = os.path.join(DATA_ROOT, "png")               # DICOM'dan üretilen PNG'ler
ORIGINALS_DIR = os.path.join(DATA_ROOT, "originals")   # etiketli orijinal görüntüler
MASKS_DIR = os.path.join(DATA_ROOT, "masks")           # ikili maskeler
RED_MASKS_DIR = os.path.join(DATA_ROOT, "red_masks")   # kırmızı maskeler (ara adım)
LABELS_DIR = os.path.join(DATA_ROOT, "labels")         # YOLO poligon etiketleri
DATASET_DIR = os.path.join(DATA_ROOT, "dataset")       # train/val ayrımı

ANNOTATIONS_JSON = os.path.join(DATA_ROOT, "annotations.json")

# ------------------------------------------------------------- parametreler
VAL_RATIO = 0.2        # doğrulama kümesi oranı
SEED = 42              # tekrarlanabilirlik
SLICE_RANGE = None     # (ilk, son) verilirse yalnızca o kesit aralığı çevrilir

for _d in (PNG_DIR, ORIGINALS_DIR, MASKS_DIR, RED_MASKS_DIR, LABELS_DIR):
    os.makedirs(_d, exist_ok=True)

print("Veri kökü:", DATA_ROOT)


## 1. DICOM → PNG

Arşiv alt klasörlere dağılmış olabilir; bu yüzden tüm dosyalar özyinelemeli olarak
taranır ve `InstanceNumber` etiketine göre sıralanır.

> Önceki sürümde kullanılan *"aralık dışındaki DICOM dosyalarını sil"* yaklaşımı
> veri kaybına açık olduğu için burada **seçme** (silme değil) mantığı kullanılmıştır.


In [ ]:
def dicom_series_to_png(dicom_root, out_dir, slice_range=None):
    """DICOM serisini InstanceNumber sırasına göre PNG'ye çevirir.

    Args:
        dicom_root:  DICOM dosyalarını barındıran klasör (alt klasörler taranır).
        out_dir:     PNG'lerin yazılacağı klasör.
        slice_range: (ilk, son) 1 tabanlı indeks çifti; verilirse yalnızca bu
                     aralıktaki kesitler çevrilir. None ise tümü çevrilir.

    Returns:
        Yazılan PNG dosyalarının yolları (kesit sırasına göre).
    """
    os.makedirs(out_dir, exist_ok=True)

    paths = []
    for root, _dirs, files in os.walk(dicom_root):
        for name in files:
            if not name.startswith("."):
                paths.append(os.path.join(root, name))
    if not paths:
        raise FileNotFoundError(f"DICOM dosyası bulunamadı: {dicom_root}")

    def instance_number(path):
        """InstanceNumber ile sıralama anahtarı; okunamayan dosya en sona gider."""
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True)
            return int(ds.get("InstanceNumber", 0) or 0)
        except Exception:
            return 10 ** 9

    paths.sort(key=instance_number)

    if slice_range is not None:
        first, last = slice_range
        paths = paths[first - 1:last]

    written = []
    for path in paths:
        ds = pydicom.dcmread(path)
        frame = Image.fromarray(ds.pixel_array)
        out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(path))[0] + ".png")
        frame.save(out_path, "PNG")
        written.append(out_path)

    print(f"{len(written)} kesit PNG'ye çevrildi -> {out_dir}")
    return written


In [ ]:
png_files = dicom_series_to_png(DICOM_ROOT, PNG_DIR, slice_range=SLICE_RANGE)
print("Örnek:", [os.path.basename(p) for p in png_files[:3]])


## 2. `annotations.json` → orijinaller ve ikili maskeler

Her görüntü için:

1. **orijinal** görüntünün bir kopyası `originals/` altına yazılır,
2. `segmentation` poligonları doldurularak **ikili maske** üretilip `masks/` altına
   `<ad>_mask.png` olarak kaydedilir.

Anotasyonu olmayan (normal) görüntüler bu adımda atlanır — bu görüntüler yalnızca
sınıflandırma görevinde kullanılır.


In [ ]:
def polygons_to_mask(polygons, size):
    """Düz koordinat listelerinden ikili (0/255) maske üretir.

    Args:
        polygons: Her biri [x1, y1, x2, y2, ...] biçiminde düz koordinat listesi.
        size:     (genişlik, yükseklik).
    """
    mask = Image.new("L", size, 0)
    draw = ImageDraw.Draw(mask)
    for flat in polygons:
        points = [(flat[i], flat[i + 1]) for i in range(0, len(flat) - 1, 2)]
        if len(points) >= 3:                     # poligon için en az 3 nokta gerekir
            draw.polygon(points, fill=255)
    return mask


def build_originals_and_masks(annotations_json, image_dir, originals_dir, masks_dir):
    """Anotasyonlu görüntüler için orijinal kopya ve ikili maske yazar.

    Returns:
        (yazılan görüntü sayısı, atlanan görüntü sayısı)
    """
    with open(annotations_json, encoding="utf-8") as f:
        data = json.load(f)

    # image_id -> poligon listesi
    polygons_by_image = {}
    for ann in data["annotations"]:
        polygons_by_image.setdefault(ann["image_id"], []).extend(ann["segmentation"])

    written = skipped = 0
    for info in data["images"]:
        polygons = polygons_by_image.get(info["id"])
        src_path = os.path.join(image_dir, info["file_name"])
        if not polygons or not os.path.exists(src_path):
            skipped += 1
            continue

        image = Image.open(src_path)
        image.save(os.path.join(originals_dir, info["file_name"]))

        mask = polygons_to_mask(polygons, image.size)
        mask.save(os.path.join(masks_dir, info["file_name"].replace(".png", "_mask.png")))
        written += 1

    print(f"{written} görüntü işlendi, {skipped} görüntü atlandı (anotasyon yok veya dosya yok).")
    return written, skipped


In [ ]:
n_written, n_skipped = build_originals_and_masks(
    ANNOTATIONS_JSON, PNG_DIR, ORIGINALS_DIR, MASKS_DIR
)
print("originals/:", len(os.listdir(ORIGINALS_DIR)), "| masks/:", len(os.listdir(MASKS_DIR)))


## 3. Maskeyi kırmızıya boyama

YOLO etiketleri maskenin **konturlarından** üretilir. Kontur çıkarımını tek kanallı
görüntüde yapmak yerine, maske üç kanallı hâle getirilip beyaz pikseller kırmızıya
boyanır; böylece ara çıktı gözle de denetlenebilir.


In [ ]:
def convert_masks_to_red(masks_dir, out_dir):
    """İkili maskelerdeki beyaz pikselleri kırmızıya çevirip kaydeder."""
    os.makedirs(out_dir, exist_ok=True)

    converted = 0
    for name in sorted(os.listdir(masks_dir)):
        if not name.endswith(".png"):
            continue
        mask = cv2.imread(os.path.join(masks_dir, name), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        color = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        color[mask == 255] = (0, 0, 255)          # BGR: kırmızı
        cv2.imwrite(os.path.join(out_dir, name), color)
        converted += 1

    print(f"{converted} maske kırmızıya çevrildi -> {out_dir}")
    return converted


convert_masks_to_red(MASKS_DIR, RED_MASKS_DIR)


## 4. Maske → YOLO poligon etiketi

YOLO segmentasyon etiket biçimi, satır başına bir nesne olacak şekilde:

```
<class_id> <x1> <y1> <x2> <y2> ... <xn> <yn>
```

Koordinatlar görüntü genişliğine/yüksekliğine bölünerek **0–1 aralığına** normalize
edilir. Tek sınıf olduğu için `class_id = 0` (`Cancer`).

> Önceki sürümde etiket adları `<ad>_mask.txt` olarak üretiliyor, ardından ayrı bir
> hücrede `_mask` eki geri alınıyordu. Burada ad doğrudan `<ad>.txt` yazılır.


In [ ]:
def mask_to_yolo_label(mask_path, label_path):
    """İkili maskeyi YOLO poligon etiketine dönüştürür.

    Returns:
        Yazılan poligon sayısı.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(mask_path)

    height, width = mask.shape
    _, binary = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    lines = []
    for contour in contours:
        if len(contour) < 3:
            continue
        points = contour.reshape(-1, 2).astype(float)
        coords = np.concatenate([points[:, 0] / width, points[:, 1] / height])
        lines.append("0 " + " ".join(f"{v:.6f}" for v in coords))

    with open(label_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
        if lines:
            f.write("\n")
    return len(lines)


In [ ]:
os.makedirs(LABELS_DIR, exist_ok=True)

total_polygons = 0
for name in sorted(os.listdir(RED_MASKS_DIR)):
    if not name.endswith(".png"):
        continue
    label_name = name.replace("_mask.png", ".txt")
    total_polygons += mask_to_yolo_label(
        os.path.join(RED_MASKS_DIR, name), os.path.join(LABELS_DIR, label_name)
    )

print(f"{len(os.listdir(LABELS_DIR))} etiket dosyası, toplam {total_polygons} lezyon poligonu")


## 5. Eğitim / doğrulama ayrımı

Görüntü ve etiket dosyaları **tek bir anahtar listesi** üzerinden bölünür.

> **Önceki sürümdeki hata:** görüntüler ve etiketler birbirinden bağımsız olarak
> `train_test_split(..., random_state=42)` ile iki kez bölünüyordu. Dosya adları
> farklı olduğu için (`x.png` ve `x_mask.png`) iki liste aynı sırada olmak zorunda
> değildir; bu durumda bir görüntü eğitim kümesine, etiketi doğrulama kümesine
> düşebilir. Bölme artık tek anahtar listesi üzerinden yapılır.


In [ ]:
def split_dataset(originals_dir, labels_dir, dataset_dir, val_ratio=0.2, seed=42):
    """Görüntü ve etiketleri eşleşmelerini bozmadan train/val klasörlerine kopyalar.

    Returns:
        (eğitim anahtarları, doğrulama anahtarları)
    """
    for sub in ("images/train", "images/val", "labels/train", "labels/val"):
        os.makedirs(os.path.join(dataset_dir, sub), exist_ok=True)

    stems = sorted(
        os.path.splitext(name)[0]
        for name in os.listdir(originals_dir)
        if name.endswith(".png")
    )
    # Etiketi bulunmayan görüntüleri dışarıda bırak
    stems = [s for s in stems if os.path.exists(os.path.join(labels_dir, s + ".txt"))]

    train_stems, val_stems = train_test_split(stems, test_size=val_ratio, random_state=seed)

    for split, items in (("train", train_stems), ("val", val_stems)):
        for stem in items:
            shutil.copy(
                os.path.join(originals_dir, stem + ".png"),
                os.path.join(dataset_dir, "images", split, stem + ".png"),
            )
            shutil.copy(
                os.path.join(labels_dir, stem + ".txt"),
                os.path.join(dataset_dir, "labels", split, stem + ".txt"),
            )

    print(f"train: {len(train_stems)} görüntü | val: {len(val_stems)} görüntü")
    return train_stems, val_stems


train_stems, val_stems = split_dataset(
    ORIGINALS_DIR, LABELS_DIR, DATASET_DIR, val_ratio=VAL_RATIO, seed=SEED
)


## 6. Doğrulama


In [ ]:
def summarize_dataset(dataset_dir):
    """Veri kümesi özetini yazdırır ve görüntü/etiket eşleşmesini doğrular."""
    for split in ("train", "val"):
        images_dir = os.path.join(dataset_dir, "images", split)
        labels_dir = os.path.join(dataset_dir, "labels", split)

        images = {os.path.splitext(n)[0] for n in os.listdir(images_dir)}
        labels = {os.path.splitext(n)[0] for n in os.listdir(labels_dir)}
        assert images == labels, f"{split}: görüntü ve etiket adları eşleşmiyor"

        polygons = sum(
            1
            for name in os.listdir(labels_dir)
            if open(os.path.join(labels_dir, name), encoding="utf-8").read().strip()
        )
        print(f"{split:>5}: {len(images):>4} görüntü | {len(labels):>4} etiket | "
              f"{polygons:>4} etiketli görüntü")

    print("Görüntü/etiket eşleşmesi doğrulandı.")


summarize_dataset(DATASET_DIR)


## Notlar ve sınırlamalar

* **Ayrım düzeyi:** Bölme görüntü (kesit) düzeyinde yapılmıştır. Aynı hastanın kesitleri
  hem eğitim hem doğrulama kümesine düşebildiği için raporlanan doğruluk iyimserdir.
  Hasta düzeyinde (group-wise) bölme, yayına hazır bir değerlendirme için gereklidir.
* **Kontur yaklaşımı:** Etiketler maskenin konturlarından üretilir. Maske zaten
  poligonlardan üretildiği için bu adım kayıpsızdır; ancak çok küçük (<3 noktalı)
  konturlar atılır.
* **Sınıf dengesizliği:** Veri kümesinde normal ve anormal kesit sayısı eşit değildir.
  Değerlendirmede yalnızca doğruluğa bakmak yanıltıcıdır; kesinlik/duyarlılık ve
  eğri altındaki alan (AUC) da raporlanmalıdır.
* **Tek sınıf:** Tüm lezyonlar `Cancer` başlığı altında toplanmıştır; lezyon tipi
  (nodül, kitle, konsolidasyon) ayrımı yapılmamıştır.
